# D1 Synthetic Forecast Skill Diagnostic

This is a diagnostic-only notebook. It does not modify the ML panel, the
feature registry, or any other artifact. It measures whether the synthetic
weather forecast in this project carries information about target-day
weather beyond what is already in origin-safe historical features. The
output is intended to inform later improvements; this notebook does not
propose or implement them.

This notebook is **not** part of the 01-07 pipeline. It lives under
`notebooks/diagnostics/` and is read-only with respect to upstream files.


## Inputs and Outputs

Inputs (read-only):

- `data/processed/ml_forecast_panel_full.parquet` (final ML panel from notebook 06).
- `outputs/ml_panel/ml_panel_feature_registry.csv` (feature registry from notebook 06), if available.
- `outputs/ml_models/benchmark_predictions_quick.parquet` and `benchmark_metrics_quick.csv` (read-only), if a quick-mode benchmark has been run.

Outputs (CSV only, no plots):

- `outputs/diagnostics/d1_forecast_skill_correlations.csv`
- `outputs/diagnostics/d1_skill_margin.csv`
- `outputs/diagnostics/d1_ensemble_spread_audit.csv`
- `outputs/diagnostics/d1_closed_flag_audit.csv`
- `outputs/diagnostics/d1_category_context.csv`


## Methodology

For each forecast variable in `{temp, wind, precip, cloud, spread}` and each horizon in `{1, 2, 3, 5, 7, 10}`, three correlations are computed on the full panel using both Pearson and Spearman:

1. `corr(forecast, realised target-day)` measures forecast skill.
2. `corr(forecast, origin-safe lag-1 observed)` measures redundancy with persistence.
3. `corr(origin-safe lag-1 observed, realised target-day)` measures the persistence baseline.

Pairs:

| Variable | Forecast | Realised target-day | Origin-safe lag-1 |
|---|---|---|---|
| temp | `temp_fcst` | `temp_mean` | `temp_obs_lag_1_origin` |
| wind | `wind_fcst` | `wind_mean` | `wind_obs_lag_1_origin` |
| precip | `precip_fcst` | `precip_val` | `precip_obs_lag_1_origin` |
| cloud | `cloud_fcst` | `cloud_mean` | `cloud_obs_lag_1_origin` |
| spread | `spread_fcst` | (requires realised dewpoint) | (requires `spread_obs_lag_1_origin`) |

Spread is skipped because the ML panel does not contain realised dewpoint, realised spread, or a `spread_obs_lag_1_origin` historical column. The spec explicitly forbids constructing a synthetic spread comparison.

Additional diagnostics computed below: skill margin = `corr(forecast, realised) - corr(persistence, realised)`; ensemble-spread variance check by `(variable, season, horizon)`; `Closed`-flag construction audit; per-category sample-size context.


## Setup and Path Discovery

In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from inside the project folder."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_ML_PANEL_DIR = OUTPUT_DIR / "ml_panel"
OUTPUT_ML_MODELS_DIR = OUTPUT_DIR / "ml_models"
OUTPUT_DIAGNOSTICS_DIR = OUTPUT_DIR / "diagnostics"

ML_PANEL_PATH = PROCESSED_DIR / "ml_forecast_panel_full.parquet"
FEATURE_REGISTRY_PATH = OUTPUT_ML_PANEL_DIR / "ml_panel_feature_registry.csv"
BENCHMARK_METRICS_QUICK_PATH = OUTPUT_ML_MODELS_DIR / "benchmark_metrics_quick.csv"
BENCHMARK_PREDICTIONS_QUICK_PATH = OUTPUT_ML_MODELS_DIR / "benchmark_predictions_quick.parquet"

OUTPUT_DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)


def project_relative(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return str(path)


print(f"Project root: {PROJECT_ROOT}")
print(f"ML panel: {project_relative(ML_PANEL_PATH)} (exists={ML_PANEL_PATH.exists()})")
print(f"Feature registry: {project_relative(FEATURE_REGISTRY_PATH)} (exists={FEATURE_REGISTRY_PATH.exists()})")
print(f"Benchmark metrics (quick): {project_relative(BENCHMARK_METRICS_QUICK_PATH)} (exists={BENCHMARK_METRICS_QUICK_PATH.exists()})")
print(f"Benchmark predictions (quick): {project_relative(BENCHMARK_PREDICTIONS_QUICK_PATH)} (exists={BENCHMARK_PREDICTIONS_QUICK_PATH.exists()})")
print(f"Diagnostics output folder: {project_relative(OUTPUT_DIAGNOSTICS_DIR)}")


## Locate Panel and Confirm Registry-Driven Column Roles

In [ ]:
import pyarrow.parquet as pq

if not ML_PANEL_PATH.exists():
    raise FileNotFoundError(
        f"Required ML panel not found at {project_relative(ML_PANEL_PATH)}. "
        "Run notebook 06 first."
    )

schema_names = list(pq.ParquetFile(ML_PANEL_PATH).schema_arrow.names)
print(f"Panel columns: {len(schema_names)} (expected 123)")

# Try to confirm column roles via the registry, if available.
registry = None
if FEATURE_REGISTRY_PATH.exists():
    registry = pd.read_csv(FEATURE_REGISTRY_PATH)
    print(f"Feature registry rows: {len(registry)}")
else:
    print("Feature registry not found; proceeding with column-name conventions only.")

# Variable specification: forecast / realised / lag-1 column names.
# Spread is intentionally skipped (no realised dewpoint or spread_obs_lag_1).
VARIABLE_SPEC = {
    "temp":   {"fcst": "temp_fcst",   "realised": "temp_mean",     "lag1": "temp_obs_lag_1_origin"},
    "wind":   {"fcst": "wind_fcst",   "realised": "wind_mean",     "lag1": "wind_obs_lag_1_origin"},
    "precip": {"fcst": "precip_fcst", "realised": "precip_val",    "lag1": "precip_obs_lag_1_origin"},
    "cloud":  {"fcst": "cloud_fcst",  "realised": "cloud_mean",    "lag1": "cloud_obs_lag_1_origin"},
    "spread": {"fcst": "spread_fcst", "realised": None,            "lag1": None},  # skipped
}

skipped_diagnostics_log = []


def column_or_skip(name, role, variable):
    if name is None or name not in schema_names:
        skipped_diagnostics_log.append({
            "variable": variable, "role": role, "missing_column": str(name),
            "reason": "column_not_in_panel",
        })
        return None
    return name


for var, spec in VARIABLE_SPEC.items():
    for role in ["fcst", "realised", "lag1"]:
        column_or_skip(spec[role], role, var)

print()
print("Skipped pairs (will be reported in CSVs as status):")
for row in skipped_diagnostics_log:
    print(f"  {row}")


## Load ML Panel With Selected Columns

Memory-conscious: only the columns needed for the diagnostics are read.


In [ ]:
HORIZONS_TO_RUN = [1, 2, 3, 5, 7, 10]

needed = set(["AvdelingID", "Analyse_Kategori", "target_date", "origin_date", "horizon", "season",
              "Closed", "is_open", "Antall_total"])
for var, spec in VARIABLE_SPEC.items():
    for role in ["fcst", "realised", "lag1"]:
        col = spec[role]
        if col and col in schema_names:
            needed.add(col)

# Ensemble spread columns
for var in ["temp", "wind", "precip", "cloud"]:
    sd_col = f"{var}_fcst_sd"
    if sd_col in schema_names:
        needed.add(sd_col)

needed_columns = sorted(c for c in needed if c in schema_names)
panel = pd.read_parquet(ML_PANEL_PATH, columns=needed_columns)
panel["horizon"] = panel["horizon"].astype("int8")

# Restrict to the horizons of interest for correlation diagnostics.
panel_corr = panel.loc[panel["horizon"].isin(HORIZONS_TO_RUN)].copy()

print(f"Loaded panel rows: {len(panel):,}")
print(f"Rows used for correlation diagnostics (horizons {HORIZONS_TO_RUN}): {len(panel_corr):,}")
print(f"Columns loaded: {len(needed_columns)}")
print(f"Memory (approx): {panel.memory_usage(deep=True).sum()/1_000_000_000:.3f} GB")


## A. Three-Correlation Block and Skill Margin

For each variable and horizon: Pearson and Spearman correlation between
the forecast and the realised target-day weather, between the forecast and
the origin-safe lag-1 observed weather, and between the lag-1 and the
realised target-day weather. Skill margin is `corr(forecast, realised) - corr(persistence, realised)` per variable and horizon.


In [ ]:
def safe_corr(s1, s2, method):
    if s1 is None or s2 is None:
        return float("nan"), 0
    pair = pd.concat([s1, s2], axis=1).dropna()
    if len(pair) < 2:
        return float("nan"), int(len(pair))
    if pair.iloc[:, 0].std(ddof=0) == 0 or pair.iloc[:, 1].std(ddof=0) == 0:
        return float("nan"), int(len(pair))
    if method == "pearson":
        return float(pair.iloc[:, 0].corr(pair.iloc[:, 1], method="pearson")), int(len(pair))
    if method == "spearman":
        return float(pair.iloc[:, 0].corr(pair.iloc[:, 1], method="spearman")), int(len(pair))
    raise ValueError(method)


def col_or_none(df, name):
    return df[name] if (name is not None and name in df.columns) else None


corr_records = []
margin_records = []

for var, spec in VARIABLE_SPEC.items():
    fcst_name, real_name, lag1_name = spec["fcst"], spec["realised"], spec["lag1"]
    fcst_avail = fcst_name in panel_corr.columns if fcst_name else False
    real_avail = real_name in panel_corr.columns if real_name else False
    lag1_avail = lag1_name in panel_corr.columns if lag1_name else False

    for h in HORIZONS_TO_RUN:
        sub = panel_corr.loc[panel_corr["horizon"] == h]

        # Forecast vs realised
        if fcst_avail and real_avail:
            r_pe, n_fr = safe_corr(sub[fcst_name], sub[real_name], "pearson")
            r_sp, _    = safe_corr(sub[fcst_name], sub[real_name], "spearman")
            status_fr = "ok"
        else:
            r_pe = r_sp = float("nan"); n_fr = 0
            status_fr = "skipped_missing_columns"

        # Forecast vs lag-1
        if fcst_avail and lag1_avail:
            r_pe_fl, n_fl = safe_corr(sub[fcst_name], sub[lag1_name], "pearson")
            r_sp_fl, _    = safe_corr(sub[fcst_name], sub[lag1_name], "spearman")
            status_fl = "ok"
        else:
            r_pe_fl = r_sp_fl = float("nan"); n_fl = 0
            status_fl = "skipped_missing_columns"

        # Lag-1 vs realised
        if lag1_avail and real_avail:
            r_pe_lr, n_lr = safe_corr(sub[lag1_name], sub[real_name], "pearson")
            r_sp_lr, _    = safe_corr(sub[lag1_name], sub[real_name], "spearman")
            status_lr = "ok"
        else:
            r_pe_lr = r_sp_lr = float("nan"); n_lr = 0
            status_lr = "skipped_missing_columns"

        corr_records.append({
            "variable": var,
            "horizon": h,
            "forecast_column": fcst_name,
            "realised_column": real_name,
            "lag1_column": lag1_name,
            "n_fcst_vs_realised": n_fr,
            "pearson_fcst_vs_realised": r_pe,
            "spearman_fcst_vs_realised": r_sp,
            "status_fcst_vs_realised": status_fr,
            "n_fcst_vs_lag1": n_fl,
            "pearson_fcst_vs_lag1": r_pe_fl,
            "spearman_fcst_vs_lag1": r_sp_fl,
            "status_fcst_vs_lag1": status_fl,
            "n_lag1_vs_realised": n_lr,
            "pearson_lag1_vs_realised": r_pe_lr,
            "spearman_lag1_vs_realised": r_sp_lr,
            "status_lag1_vs_realised": status_lr,
        })

        # Skill margin: forecast vs persistence relative to realised target.
        if status_fr == "ok" and status_lr == "ok":
            margin = r_pe - r_pe_lr
            margin_status = "ok"
        else:
            margin = float("nan")
            margin_status = "skipped_missing_inputs"
        margin_records.append({
            "variable": var,
            "horizon": h,
            "n": min(n_fr, n_lr) if (status_fr == "ok" and status_lr == "ok") else 0,
            "corr_forecast_realised_pearson": r_pe,
            "corr_persistence_realised_pearson": r_pe_lr,
            "skill_margin_pearson": margin,
            "status": margin_status,
        })

correlations_df = pd.DataFrame(corr_records)
skill_margin_df = pd.DataFrame(margin_records)

print(f"Correlation rows: {len(correlations_df)}")
print(f"Skill margin rows: {len(skill_margin_df)}")

display_cols = [
    "variable", "horizon",
    "pearson_fcst_vs_realised", "pearson_fcst_vs_lag1", "pearson_lag1_vs_realised",
]
display(correlations_df[display_cols].round(4))
display(skill_margin_df.round(4))


## B. Ensemble Spread Variance Check

For each `*_fcst_sd` column, compute the standard deviation of the
ensemble spread within each `(season, horizon)` cell. If the within-cell
std is near zero relative to the cell mean, the ensemble spread is
essentially a `(season, horizon)` constant and carries no per-row
uncertainty information. Cells with `ratio < 0.05` are flagged.


In [ ]:
SD_VARIABLES = []
for var in ["temp", "wind", "precip", "cloud"]:
    sd_col = f"{var}_fcst_sd"
    if sd_col in panel.columns:
        SD_VARIABLES.append((var, sd_col))
    else:
        skipped_diagnostics_log.append({
            "variable": var, "role": "ensemble_sd", "missing_column": sd_col,
            "reason": "column_not_in_panel",
        })

spread_records = []
for var, sd_col in SD_VARIABLES:
    grouped = panel.groupby(["season", "horizon"], observed=True)[sd_col]
    agg = grouped.agg(mean_sd="mean", within_cell_std_of_sd="std", n="size").reset_index()
    agg["variable"] = var
    agg["ratio"] = np.where(agg["mean_sd"] > 0, agg["within_cell_std_of_sd"] / agg["mean_sd"], np.nan)
    agg["ratio_below_0_05_flag"] = (agg["ratio"] < 0.05).astype("int8")
    spread_records.append(agg[["variable", "season", "horizon", "mean_sd",
                               "within_cell_std_of_sd", "ratio", "n", "ratio_below_0_05_flag"]])

ensemble_spread_audit_df = (
    pd.concat(spread_records, ignore_index=True)
    if spread_records else pd.DataFrame(columns=["variable", "season", "horizon",
                                                 "mean_sd", "within_cell_std_of_sd", "ratio",
                                                 "n", "ratio_below_0_05_flag"])
)

print(f"Ensemble-spread audit rows: {len(ensemble_spread_audit_df)}")
print(f"Cells flagged with ratio < 0.05: {int(ensemble_spread_audit_df['ratio_below_0_05_flag'].sum())} of {len(ensemble_spread_audit_df)}")
display(ensemble_spread_audit_df.round(6).head(40))


## C. Closed-Flag Construction Audit

Reports four quantities to characterise the `Closed` flag without
modifying it: shares of contradicting / consistent rows, and correlations
of `Closed` with `is_open` and `Antall_total`. A near-perfect alignment
of `Closed == 1` with zero-sales rows and a strong negative correlation
with `Antall_total` would indicate the flag is sales-derived rather than
an independent operational source.


In [ ]:
closed_records = []

if {"Closed", "is_open", "Antall_total"}.issubset(panel.columns):
    n_total = int(len(panel))
    closed_eq_1 = panel["Closed"] == 1
    closed_eq_0 = panel["Closed"] == 0
    sales_pos = panel["Antall_total"] > 0
    sales_zero = panel["Antall_total"] == 0

    share_closed_1_sales_pos = float((closed_eq_1 & sales_pos).mean())
    share_closed_0_sales_zero = float((closed_eq_0 & sales_zero).mean())
    share_closed_eq_1 = float(closed_eq_1.mean())
    share_sales_zero = float(sales_zero.mean())
    corr_closed_is_open = float(panel["Closed"].astype("float64").corr(panel["is_open"].astype("float64")))
    corr_closed_antall = float(panel["Closed"].astype("float64").corr(panel["Antall_total"].astype("float64")))

    closed_records.append(("share_closed_eq_1_AND_Antall_total_gt_0", share_closed_1_sales_pos))
    closed_records.append(("share_closed_eq_0_AND_Antall_total_eq_0", share_closed_0_sales_zero))
    closed_records.append(("corr_closed_is_open", corr_closed_is_open))
    closed_records.append(("corr_closed_antall_total", corr_closed_antall))
    closed_records.append(("share_closed_eq_1_overall", share_closed_eq_1))
    closed_records.append(("share_sales_eq_0_overall", share_sales_zero))
    closed_records.append(("n_total", n_total))

    print("Closed-flag diagnostics:")
    for k, v in closed_records:
        print(f"  {k} = {v}")
else:
    skipped_diagnostics_log.append({
        "variable": "Closed", "role": "closed_flag_audit",
        "missing_column": "Closed/is_open/Antall_total",
        "reason": "required_columns_missing",
    })
    print("Closed-flag audit skipped because one or more required columns are missing.")

closed_flag_audit_df = pd.DataFrame(closed_records, columns=["metric", "value"])
display(closed_flag_audit_df)


## D. Per-Category Sample-Size Context

Counts and basic distributions per `Analyse_Kategori`, plus an optional
share of zero predictions if the quick-mode benchmark predictions parquet
is available. Zero predictions in the saved benchmark file may be either
genuine zero-sales predictions or post-clip artefacts; the exact pre-clip
count cannot be recovered from the saved file.


In [ ]:
if "Analyse_Kategori" not in panel.columns:
    skipped_diagnostics_log.append({
        "variable": "Analyse_Kategori", "role": "category_context",
        "missing_column": "Analyse_Kategori", "reason": "required_column_missing",
    })
    cat_df = pd.DataFrame(columns=["Analyse_Kategori", "n_rows", "mean_antall_open_only", "share_zero_sales"])
else:
    open_mask = panel["is_open"] == 1 if "is_open" in panel.columns else None
    grouped = panel.groupby("Analyse_Kategori", observed=True)
    n_rows = grouped.size().rename("n_rows")
    share_zero = grouped["Antall_total"].apply(lambda s: float((s == 0).mean())).rename("share_zero_sales")
    if open_mask is not None:
        mean_open = (
            panel.loc[open_mask]
            .groupby("Analyse_Kategori", observed=True)["Antall_total"]
            .mean()
            .rename("mean_antall_open_only")
        )
    else:
        mean_open = grouped["Antall_total"].mean().rename("mean_antall_open_only")
    cat_df = pd.concat([n_rows, mean_open, share_zero], axis=1).reset_index()

# Optional: share of predictions equal to zero per category from the quick benchmark, as a proxy for clipped predictions.
share_pred_zero = None
if BENCHMARK_PREDICTIONS_QUICK_PATH.exists():
    try:
        pred_cols = ["Analyse_Kategori", "y_pred", "model_family", "feature_set"]
        bench_pred = pd.read_parquet(BENCHMARK_PREDICTIONS_QUICK_PATH, columns=pred_cols)
        share_pred_zero = (
            bench_pred.assign(pred_zero=(bench_pred["y_pred"] <= 0).astype("int8"))
            .groupby("Analyse_Kategori", observed=True)["pred_zero"].mean()
            .rename("share_pred_zero_or_clipped_quick")
        )
        del bench_pred
        gc.collect()
    except Exception as exc:
        skipped_diagnostics_log.append({
            "variable": "benchmark_predictions_quick",
            "role": "category_context",
            "missing_column": str(BENCHMARK_PREDICTIONS_QUICK_PATH.name),
            "reason": f"read_failed: {type(exc).__name__}: {exc}",
        })

if share_pred_zero is not None:
    cat_df = cat_df.merge(share_pred_zero, on="Analyse_Kategori", how="left")
else:
    cat_df["share_pred_zero_or_clipped_quick"] = np.nan

category_context_df = cat_df.sort_values("Analyse_Kategori").reset_index(drop=True)
print(f"Per-category context rows: {len(category_context_df)}")
display(category_context_df.round(4))


## Save All Diagnostic CSVs

In [ ]:
# Append a status column to correlations_df / skill_margin_df reflecting the skipped log.
skipped_log_df = pd.DataFrame(skipped_diagnostics_log)
print(f"Skipped diagnostics log rows: {len(skipped_log_df)}")
if len(skipped_log_df) > 0:
    display(skipped_log_df)

CORRELATIONS_PATH = OUTPUT_DIAGNOSTICS_DIR / "d1_forecast_skill_correlations.csv"
SKILL_MARGIN_PATH = OUTPUT_DIAGNOSTICS_DIR / "d1_skill_margin.csv"
SPREAD_AUDIT_PATH = OUTPUT_DIAGNOSTICS_DIR / "d1_ensemble_spread_audit.csv"
CLOSED_AUDIT_PATH = OUTPUT_DIAGNOSTICS_DIR / "d1_closed_flag_audit.csv"
CATEGORY_CONTEXT_PATH = OUTPUT_DIAGNOSTICS_DIR / "d1_category_context.csv"

correlations_df.to_csv(CORRELATIONS_PATH, index=False)
skill_margin_df.to_csv(SKILL_MARGIN_PATH, index=False)
ensemble_spread_audit_df.to_csv(SPREAD_AUDIT_PATH, index=False)
closed_flag_audit_df.to_csv(CLOSED_AUDIT_PATH, index=False)
category_context_df.to_csv(CATEGORY_CONTEXT_PATH, index=False)

for p in [CORRELATIONS_PATH, SKILL_MARGIN_PATH, SPREAD_AUDIT_PATH, CLOSED_AUDIT_PATH, CATEGORY_CONTEXT_PATH]:
    print(f"Wrote: {project_relative(p)} (rows={sum(1 for _ in p.open()) - 1})")


## Diagnostic Summary

The bullet points below describe what each diagnostic measures and how to read it. They are intentionally descriptive only; the exact numerical values live in the saved CSVs.

- The three-correlation table (`d1_forecast_skill_correlations.csv`) shows, per variable and horizon, how strongly the synthetic forecast tracks the realised target-day weather, how closely it tracks origin-safe lag-1 observed weather (i.e. the persistence baseline), and how strongly persistence itself tracks the realised target. Both Pearson and Spearman columns are reported.
- The skill-margin table (`d1_skill_margin.csv`) is `corr(forecast, realised) - corr(persistence, realised)` per variable and horizon. Positive values mean the forecast carries information beyond persistence; values near zero or negative mean the forecast is no better than yesterday's weather.
- Spread is intentionally absent from the correlation tables. Realised dewpoint, realised spread, and `spread_obs_lag_1_origin` are not present in the ML panel, so a like-for-like comparison cannot be made; constructing a synthetic comparison was explicitly out of scope.
- The ensemble-spread audit (`d1_ensemble_spread_audit.csv`) asks whether `*_fcst_sd` carries per-row information. For every `(variable, season, horizon)` cell we compute the within-cell standard deviation of `*_fcst_sd` divided by the cell mean. A ratio below `0.05` is flagged as degenerate — it indicates that `*_fcst_sd` is effectively a `(season, horizon)` constant across rows.
- The closed-flag audit (`d1_closed_flag_audit.csv`) reports the share of contradicting rows (`Closed == 1` with positive sales, or `Closed == 0` with zero sales), the overall closed share, the overall zero-sales share, and the correlations of `Closed` with `is_open` and with `Antall_total`. A near-zero contradiction rate combined with a strong negative correlation between `Closed` and `Antall_total` is consistent with `Closed` being derived from sales rather than an independent operational source.
- The per-category context (`d1_category_context.csv`) gives row counts per `Analyse_Kategori`, the mean of `Antall_total` on open days only, the share of zero-sales rows, and (where the quick-mode benchmark predictions exist) the share of predictions equal to zero. The last column is a coarse proxy for clipping; the exact pre-clip share cannot be recovered from the saved benchmark predictions parquet because they are stored post-clip.
- Skipped diagnostics, if any, are listed in the cell above the CSV writes. Empty skip log means every requested correlation pair and audit was computed.
